# Resume ISLES'24 NCCT Fold 4 on Kaggle

Before running:

1. Create a Kaggle Notebook with **GPU T4 x2**.
2. Turn **Internet ON**.
3. Add the private Dataset `isles2024-ncct-fold4-resume` as an Input.
4. Run all cells.

The notebook intentionally exposes only **one T4 GPU** to nnU-Net so the transferred single-GPU Fold 4 checkpoint continues with the same training semantics. The second T4 is left unused.


In [ ]:
import os, subprocess, sys
from pathlib import Path

subprocess.run([sys.executable,'-m','pip','install','-q','nnunetv2==2.8.1'],check=True)
subprocess.run([
    'git','clone','--depth','1',
    'https://github.com/NTinkicht/HybridStrokeSeg-Reproduction.git',
    '/kaggle/working/HybridStrokeSeg-Reproduction'
],check=True)
subprocess.run([
    sys.executable,'-m','pip','install','-q','-e',
    '/kaggle/working/HybridStrokeSeg-Reproduction'
],check=True)


## Locate the private handoff archive


In [ ]:
matches = list(Path('/kaggle/input').rglob('isles2024_fold4_resume.tar'))
print('Matches:', matches)
if len(matches) != 1:
    raise RuntimeError('Attach exactly one private ISLES2024 Fold4 resume Dataset input')
ARCHIVE = matches[0]
print('Using:', ARCHIVE)


## Verify GPU and resume Fold 4


In [ ]:
import torch
print('Visible GPUs before restriction:', torch.cuda.device_count())
for idx in range(torch.cuda.device_count()):
    print(idx, torch.cuda.get_device_name(idx))

env = os.environ.copy()
env['CUDA_VISIBLE_DEVICES'] = '0'

subprocess.run([
    sys.executable,
    '/kaggle/working/HybridStrokeSeg-Reproduction/scripts/run_kaggle_fold4_resume.py',
    '--archive',str(ARCHIVE),
    '--work-root','/kaggle/working/isles2024_runtime',
    '--output-root','/kaggle/working/isles2024_fold4_output'
],check=True,env=env)


## Final output

When the notebook finishes, Kaggle will save `/kaggle/working/isles2024_fold4_output` as notebook output. It contains Fold 4's final checkpoint and validation outputs. Keep that output private.


In [ ]:
OUT = Path('/kaggle/working/isles2024_fold4_output')
print('Output exists:', OUT.exists())
if OUT.exists():
    for p in sorted(OUT.rglob('*')):
        if p.is_file():
            print(p.relative_to(OUT), round(p.stat().st_size/(1024**2),2),'MiB')
